# Week 5 SVM & Kernel Trick

**Datasets:** Taobao User Behavior

## Topics
1. **Conversion classification (Taobao)** — predict whether a user completes at least one purchase.
2. **Kernel comparison** — linear vs. RBF SVM with grid-search tuning.


## Data loading and cleaning

Aggregate Taobao events to user-level features.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(42)
DATA_DIR = Path(".")

orders = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_orders_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_reviews_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_payments_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_customers_dataset.csv")

pay = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    n_payments=("payment_sequential", "max"),
).reset_index()
itm = items.groupby("order_id").agg(
    total_price=("price", "sum"),
    freight_value=("freight_value", "sum"),
    n_items=("order_item_id", "count"),
).reset_index()
rev = reviews.groupby("order_id").agg(review_score=("review_score", "mean")).reset_index()

olist_df = (
    orders.merge(rev, on="order_id")
    .merge(itm, on="order_id")
    .merge(pay, on="order_id")
    .merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id")
)
olist_df["order_purchase_timestamp"] = pd.to_datetime(olist_df["order_purchase_timestamp"])
olist_df["order_month"] = olist_df["order_purchase_timestamp"].dt.month
olist_df["delivery_days"] = (
    pd.to_datetime(olist_df["order_delivered_customer_date"], errors="coerce")
    - olist_df["order_purchase_timestamp"]
).dt.days
olist_df["delivery_days"] = olist_df["delivery_days"].fillna(olist_df["delivery_days"].median())
olist_df = olist_df.dropna(subset=["review_score", "payment_value", "total_price"])

ucl_raw = pd.read_excel(DATA_DIR / "ucl_dataset.xlsx")
ucl_raw = ucl_raw.rename(columns={"Invoice": "InvoiceNo", "Price": "UnitPrice", "Customer ID": "CustomerID"})
ucl_raw["InvoiceDate"] = pd.to_datetime(ucl_raw["InvoiceDate"])
ucl_raw = ucl_raw.dropna(subset=["CustomerID"])
ucl_raw = ucl_raw[~ucl_raw["InvoiceNo"].astype(str).str.startswith("C", na=False)]
ucl_raw = ucl_raw[(ucl_raw["Quantity"] > 0) & (ucl_raw["UnitPrice"] > 0)]
ucl_raw["LineTotal"] = ucl_raw["Quantity"] * ucl_raw["UnitPrice"]
anchor = ucl_raw["InvoiceDate"].max() + pd.Timedelta(days=1)
ucl_df = (
    ucl_raw.groupby("CustomerID")
    .agg(
        recency=("InvoiceDate", lambda x: (anchor - x.max()).days),
        frequency=("InvoiceNo", "nunique"),
        monetary=("LineTotal", "sum"),
        avg_basket=("LineTotal", "mean"),
        n_items=("Quantity", "sum"),
    )
    .reset_index()
)
ucl_df["log_monetary"] = np.log1p(ucl_df["monetary"])
ucl_df["log_frequency"] = np.log1p(ucl_df["frequency"])

TAOBAO_PATH = DATA_DIR / "TaobaoUserBehavior.csv"
user_set = set()
chunks = []
for chunk in pd.read_csv(
    TAOBAO_PATH,
    header=None,
    names=["user_id", "item_id", "category_id", "behavior", "timestamp"],
    chunksize=500_000,
):
    chunk = chunk[chunk["user_id"].isin(user_set) | (len(user_set) < 50000)]
    if len(user_set) < 50000:
        for u in chunk["user_id"].unique():
            if len(user_set) >= 50000:
                break
            user_set.add(u)
    chunk = chunk[chunk["user_id"].isin(user_set)]
    if len(chunk):
        chunks.append(chunk)
    if len(user_set) >= 50000 and sum(len(c) for c in chunks) > 200_000:
        break

taobao = pd.concat(chunks, ignore_index=True)
taobao_df = (
    taobao.groupby("user_id")
    .agg(
        n_events=("behavior", "count"),
        n_pv=("behavior", lambda x: (x == "pv").sum()),
        n_cart=("behavior", lambda x: (x == "cart").sum()),
        n_fav=("behavior", lambda x: (x == "fav").sum()),
        n_buy=("behavior", lambda x: (x == "buy").sum()),
        n_categories=("category_id", "nunique"),
        n_items=("item_id", "nunique"),
    )
    .reset_index()
)
taobao_df["cart_rate"] = taobao_df["n_cart"] / taobao_df["n_events"].clip(lower=1)
taobao_df["converted"] = (taobao_df["n_buy"] > 0).astype(int)

print(f"Olist orders: {len(olist_df):,}")
print(f"UCL customers: {len(ucl_df):,}")
print(f"Taobao users: {len(taobao_df):,}")


# Conversion target and features


In [ ]:
feats = ["n_pv", "n_cart", "n_fav", "n_categories", "n_items", "cart_rate"]
X = taobao_df[feats].values
y = taobao_df["converted"].values

print(f"Users: {len(taobao_df):,}")
print(f"Conversion rate: {100 * y.mean():.1f}%")
print(f"Features: {feats}")


# Linear SVM


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

linear = SVC(kernel="linear", C=1.0, class_weight="balanced", probability=True)
linear.fit(X_train_s, y_train)
linear_auc = roc_auc_score(y_test, linear.predict_proba(X_test_s)[:, 1])
linear_f1 = f1_score(y_test, linear.predict(X_test_s))

print(f"Linear SVM ROC-AUC: {linear_auc:.4f}")
print(f"Linear SVM F1: {linear_f1:.4f}")


# RBF kernel SVM


In [ ]:
rbf = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", probability=True)
rbf.fit(X_train_s, y_train)
rbf_auc = roc_auc_score(y_test, rbf.predict_proba(X_test_s)[:, 1])
rbf_f1 = f1_score(y_test, rbf.predict(X_test_s))

print(f"RBF SVM ROC-AUC: {rbf_auc:.4f}")
print(f"RBF SVM F1: {rbf_f1:.4f}")


# Grid search hyperparameter tuning


In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    SVC(class_weight="balanced", probability=True),
    {"C": [0.1, 1, 10], "kernel": ["linear", "rbf"], "gamma": ["scale", "auto"]},
    cv=3,
    scoring="roc_auc",
    n_jobs=-1,
)
grid.fit(X_train_s, y_train)
best = grid.best_estimator_
best_auc = roc_auc_score(y_test, best.predict_proba(X_test_s)[:, 1])
best_f1 = f1_score(y_test, best.predict(X_test_s))

print(f"Best params: {grid.best_params_}")
print(f"Tuned SVM ROC-AUC: {best_auc:.4f}")
print(f"Tuned SVM F1: {best_f1:.4f}")


The RBF kernel raises ROC-AUC above the linear boundary, which supports a nonlinear relationship between browsing behavior and purchase conversion.


# Kernel comparison plot


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    ["Linear SVM", "RBF SVM", "Tuned SVM"],
    [linear_auc, rbf_auc, best_auc],
    color=["#8172B3", "#CCB974", "#64B5CD"],
)
ax.set_ylabel("ROC-AUC")
ax.set_title("Week 5: SVM kernel comparison (Taobao purchase conversion)")
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()
